# 02 — Gather DeFiLlama data

## What this notebook does

This notebook collects the actual data used in the project.

We use DeFiLlama to get:
- **historical TVL** for the selected protocols
- **current APY** where DeFiLlama provides a yield page

The result will be two datasets:

1. **`staking_tvl_timeseries.csv`**  
   Main analysis dataset with one row per date and protocol

2. **`staking_protocol_summary.csv`**  
   Small summary table with current TVL and APY (if available)

Important:
- This notebook is **data-first**
- We keep metadata to a minimum
- Native staking is **not included as a DeFiLlama protocol row**, because it is a baseline mechanism rather than a normal DeFi protocol entry

In [1]:
import re
import requests
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("..")
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_RAW = PROJECT_ROOT / "data" / "raw"

## Protocols used in the data collection

These are the DeFiLlama protocol slugs we use:
- Lido => `lido`
- Rocket Pool => `rocket-pool`
- EigenLayer => `eigencloud` in DeFiLlama

In [2]:
# Here we add the protocols that we want to compare
# We also analyze native staking, but we dont add it here, because we manually add it later
protocols = [
    {"protocol": "Lido", "category": "liquid_staking", "slug": "lido", "has_yield_page": True},
    {"protocol": "Rocket Pool", "category": "liquid_staking", "slug": "rocket-pool", "has_yield_page": True},
    {"protocol": "EigenLayer", "category": "restaking", "slug": "eigencloud", "has_yield_page": False},
]

protocols_df = pd.DataFrame(protocols)
protocols_df

,protocol,category,slug,has_yield_page
0,Lido,liquid_staking,lido,True
1,Rocket Pool,liquid_staking,rocket-pool,True
2,EigenLayer,restaking,eigencloud,False


## Helper functions

In [3]:
# Get the current TVL for one protocol from DeFiLlama
def fetch_current_tvl(slug: str):
    url = f"https://api.llama.fi/tvl/{slug}"
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    value = r.json()
    if isinstance(value, (int, float)):
        return float(value), url
    return pd.NA, url

# Get the full protocol. Use to extract TVL history
def fetch_protocol_json(slug: str):
    url = f"https://api.llama.fi/protocol/{slug}"
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    return r.json(), url

# DeFiLlama can name the TVl column differently.
# Here we standardize it to date + tvl_usd
def normalize_tvl_df(df: pd.DataFrame):
    # Check for common tvl names and take the first one if exists
    possible_value_cols = ["totalLiquidityUSD", "liquidity", "totalLiquidityUsd", "tvl", "value"]
    value_col = next((c for c in possible_value_cols if c in df.columns), None)
    
    # If we dont have both a date and a TVL we return an empty df
    if "date" not in df.columns or value_col is None:
        return pd.DataFrame(columns=["date", "tvl_usd"])
    
    # Keep only the date column and the found TVL
    out = df[["date", value_col]].copy()
    # Rename for consistency
    out = out.rename(columns={value_col: "tvl_usd"})
    # Convert UNIX timetsamp to date
    out["date"] = pd.to_datetime(out["date"], unit="s", errors="coerce")
    out["tvl_usd"] = pd.to_numeric(out["tvl_usd"], errors="coerce")
    out = out.dropna(subset=["date", "tvl_usd"]).sort_values("date").reset_index(drop=True)
    return out

# Try to extract a usable TVL history
# First try the overall TVL history and if it doesnt work,
# we try chain-level histories with Ethereum as a preference
def extract_tvl_timeseries(protocol_json: dict):
    # Some protocols have a direct overall TVL history under "tvl"
    tvl_obj = protocol_json.get("tvl")
    if isinstance(tvl_obj, list) and len(tvl_obj) > 0:
        return normalize_tvl_df(pd.DataFrame(tvl_obj))
    
    # If no direct history, look into chain-level history
    chain_tvls = protocol_json.get("chainTvls", {})
    if isinstance(chain_tvls, dict) and len(chain_tvls) > 0:
        # Start with all available chains
        preferred_keys = list(chain_tvls.keys())
        # If Ethereum is present, we take that 
        if "Ethereum" in chain_tvls:
            preferred_keys = ["Ethereum"] + [k for k in preferred_keys if k != "Ethereum"]
        
        # Go through the selected chains
        for key in preferred_keys:
            entry = chain_tvls.get(key)
            # Check if chain is a dictionary
            if isinstance(entry, dict):
                # Different protocols may store the TVL history udner different nested keys
                for nested_key in ["tvl", "tokensInUsd", "tokens"]:
                    nested = entry.get(nested_key)
                    # If we find a non-empty list, normalize it
                    if isinstance(nested, list) and len(nested) > 0:
                        return normalize_tvl_df(pd.DataFrame(nested))

    return pd.DataFrame(columns=["date", "tvl_usd"])

# For protocols with a DeFiLlama yield page, try to get the displayed average APY
def fetch_average_apy(slug: str):
    url = f"https://defillama.com/protocol/yields/{slug}"
    try:
        r = requests.get(url, timeout=30)
        r.raise_for_status()
        text = r.text
        # Look for a phrase like "average APY 2.43%"
        match = re.search(r"average APY\s*([0-9]+(?:\.[0-9]+)?)%", text, flags=re.IGNORECASE)
        # If the pattern is found
        if match:
            # return the APY, url, and rate type
            return float(match.group(1)), url, "APY"
        return pd.NA, url, "APY not found on yield page"
    except Exception:
        return pd.NA, url

## Collect the data

In [4]:
timeseries_frames = []
summary_rows = []

for row in protocols:
    protocol = row["protocol"]
    category = row["category"]
    slug = row["slug"]
    has_yield_page = row["has_yield_page"]
    
    # Current TVL gives us a quick size comparison
    current_tvl, tvl_source = fetch_current_tvl(slug)
    # The protocol KSON is used to recover TVL history
    protocol_json, history_source = fetch_protocol_json(slug)
    hist_df = extract_tvl_timeseries(protocol_json)
    
    # Only get APY where a DeFiLlama yield page exists
    if has_yield_page:
        annual_rate_percent, rate_source, rate_type = fetch_average_apy(slug)
    else:
        annual_rate_percent = pd.NA
        rate_source = "No DeFiLlama yield page used"
        rate_type = pd.NA

    # Add protocol label columns
    if not hist_df.empty:
        hist_df["protocol"] = protocol
        hist_df["category"] = category
        hist_df["slug"] = slug
        timeseries_frames.append(hist_df)
    
    summary_rows.append({
        "protocol": protocol,
        "category": category,
        "slug": slug,
        "current_tvl_usd": current_tvl,
        "annual_rate_percent": annual_rate_percent,
        "rate_type": rate_type,
        "source": rate_source,
        "note": pd.NA,
        "tvl_source": tvl_source,
        "history_source": history_source
    })

staking_tvl_timeseries_df = pd.concat(timeseries_frames, ignore_index=True)
staking_protocol_summary_df = pd.DataFrame(summary_rows)

staking_tvl_timeseries_df.head()

,date,tvl_usd,protocol,category,slug
0,2020-12-19 23:00:00,1484680,Lido,liquid_staking,lido
1,2020-12-20 23:00:00,2697598,Lido,liquid_staking,lido
2,2020-12-21 23:00:00,3410253,Lido,liquid_staking,lido
3,2020-12-22 23:00:00,4563625,Lido,liquid_staking,lido
4,2020-12-23 23:00:00,4661610,Lido,liquid_staking,lido


In [5]:
# Native staking we can use for a benchmark
# We will add it manually to the table with information from ethereum.org
summary_rows.append({
    "protocol": "Ethereum Native Staking",
    "category": "native_staking",
    "slug": pd.NA,
    "current_tvl_usd": pd.NA,
    "annual_rate_percent": 2.8,
    "rate_type": "APR",
    "source": "https://ethereum.org/staking/",
    "note": "Benchmark row from ethereum.org staking page; Total ETH staked: 38,746,608; Validators: 899,017",
    "tvl_source": pd.NA,
    "history_source": pd.NA
})

# Rebuild the time series
staking_tvl_timeseries_df = (
    pd.concat(timeseries_frames, ignore_index=True)
    if timeseries_frames else
    pd.DataFrame(columns=["date", "tvl_usd", "protocol", "category", "slug"])
)

staking_protocol_summary_df = pd.DataFrame(summary_rows)

# Clean the time series
if not staking_tvl_timeseries_df.empty:
    staking_tvl_timeseries_df["tvl_usd"] = pd.to_numeric(
        staking_tvl_timeseries_df["tvl_usd"], errors="coerce"
    )
    staking_tvl_timeseries_df = staking_tvl_timeseries_df.dropna(subset=["tvl_usd"])
    staking_tvl_timeseries_df = staking_tvl_timeseries_df.sort_values(
        ["protocol", "date"]
    ).reset_index(drop=True)

# Make sure TVL is numeric
staking_protocol_summary_df["current_tvl_usd"] = pd.to_numeric(
    staking_protocol_summary_df["current_tvl_usd"], errors="coerce"
)

## Save the datasets

In [6]:
timeseries_path = DATA_PROCESSED / "staking_tvl_timeseries.csv"
summary_path = DATA_PROCESSED / "staking_protocol_summary.csv"

staking_tvl_timeseries_df.to_csv(timeseries_path, index=False)
staking_protocol_summary_df.to_csv(summary_path, index=False)

In [9]:
staking_tvl_timeseries_df

,date,tvl_usd,protocol,category,slug
0,2023-06-14 00:00:00,13300350,EigenLayer,restaking,eigencloud
1,2023-06-15 00:00:00,16484319,EigenLayer,restaking,eigencloud
2,2023-06-16 00:00:00,16557169,EigenLayer,restaking,eigencloud
3,2023-06-17 00:00:00,17099837,EigenLayer,restaking,eigencloud
4,2023-06-18 00:00:00,17177115,EigenLayer,restaking,eigencloud
...,...,...,...,...,...
4731,2026-05-14 00:00:00,1130286171,Rocket Pool,liquid_staking,rocket-pool
4732,2026-05-15 00:00:00,1137796615,Rocket Pool,liquid_staking,rocket-pool
4733,2026-05-16 00:00:00,1106988804,Rocket Pool,liquid_staking,rocket-pool
4734,2026-05-17 00:00:00,1083476389,Rocket Pool,liquid_staking,rocket-pool


## What we have now

After this notebook, we have:
- a **time-series dataset** for TVL
- a **small summary table** for current TVL and APY

This is enough to move to analysis.

The next notebook should answer questions like:
- How did TVL evolve over time for the selected protocols?
- How large is restaking compared with liquid staking?
- What APY values are available directly from DeFiLlama, and where are they missing?

# NEW PART
# 02 — Collect TVL and yield histories

This notebook collects the empirical data for the project.

Quantitative sample:

- **Liquid staking tokens:** stETH and rETH
- **Liquid restaking tokens:** eETH/weETH, ezETH and rsETH

It creates four files:

- `staking_tvl_timeseries.csv`
- `staking_yield_timeseries.csv`
- `yield_pool_candidates.csv`
- `staking_protocol_summary.csv`

The project compares liquid staking with liquid restaking. Native staking and direct restaking remain part of the conceptual taxonomy, but are not forced into this numerical dataset.

In [1]:
import re
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests

PROJECT_ROOT = Path("..")
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

TVL_API = "https://api.llama.fi"
YIELD_API = "https://yields.llama.fi"

RETRIEVED_AT = datetime.now(timezone.utc).isoformat()

SESSION = requests.Session()
SESSION.headers.update(
    {
        "User-Agent": "DeFi-yield-course-project/1.0"
    }
)

In [2]:
mapping_path = DATA_PROCESSED / "protocol_mapping.csv"

if not mapping_path.exists():
    raise FileNotFoundError(
        "Run Notebook 01 first so protocol_mapping.csv exists."
    )

protocols_df = pd.read_csv(mapping_path)

required_columns = {
    "product_id",
    "protocol",
    "category",
    "token",
    "defillama_slug",
    "yield_symbol",
}

missing_columns = required_columns.difference(protocols_df.columns)

assert not missing_columns, (
    "Notebook 01 is missing required columns: "
    f"{sorted(missing_columns)}"
)

# By default, the yield-project identifier is the same as the DeFiLlama slug.
# This column is optional in Notebook 01, so we create it here if absent.
if "yield_project" not in protocols_df.columns:
    protocols_df["yield_project"] = protocols_df["defillama_slug"]

protocols_df = protocols_df[
    [
        "product_id",
        "protocol",
        "category",
        "token",
        "defillama_slug",
        "yield_project",
        "yield_symbol",
    ]
].copy()

assert protocols_df["product_id"].is_unique
assert protocols_df["defillama_slug"].notna().all()

expected_categories = {"liquid_staking", "liquid_restaking"}

assert set(protocols_df["category"]) == expected_categories, (
    "This notebook should only contain the LST and LRT sample."
)

protocols_df

,product_id,protocol,category,token,defillama_slug,yield_project,yield_symbol
0,lido_steth,Lido,liquid_staking,stETH,lido,lido,stETH
1,rocket_pool_reth,Rocket Pool,liquid_staking,rETH,rocket-pool,rocket-pool,rETH
2,etherfi_weeth,Ether.fi,liquid_restaking,weETH,ether.fi-stake,ether.fi-stake,weETH|eETH
3,renzo_ezeth,Renzo,liquid_restaking,ezETH,renzo,renzo,ezETH
4,kelp_rseth,Kelp DAO,liquid_restaking,rsETH,kelp,kelp,rsETH


In [3]:
def get_json(url: str):
    response = SESSION.get(url, timeout=60)
    response.raise_for_status()
    return response.json()


def normalise_label(value) -> str:
    """Lowercase and remove punctuation so labels can be compared safely."""
    return re.sub(r"[^a-z0-9]", "", str(value).lower())


def parse_dates(values) -> pd.Series:
    """
    Parse API date columns whether they are:
    - Unix timestamps in seconds,
    - Unix timestamps in milliseconds,
    - ordinary date strings.
    """
    values = pd.Series(values)
    numeric_values = pd.to_numeric(values, errors="coerce")

    if numeric_values.notna().all():
        unit = "ms" if numeric_values.abs().median() > 10**11 else "s"

        return (
            pd.to_datetime(
                numeric_values,
                unit=unit,
                errors="coerce",
                utc=True,
            )
            .dt.tz_localize(None)
        )

    return (
        pd.to_datetime(
            values,
            errors="coerce",
            utc=True,
        )
        .dt.tz_localize(None)
    )


def normalise_tvl_history(records) -> pd.DataFrame:
    """Convert a DeFiLlama TVL history to standard date + tvl_usd columns."""

    if not isinstance(records, list) or not records:
        return pd.DataFrame(columns=["date", "tvl_usd"])

    df = pd.DataFrame(records)

    value_candidates = [
        "totalLiquidityUSD",
        "totalLiquidityUsd",
        "liquidity",
        "tvl",
        "value",
    ]

    value_column = next(
        (
            column
            for column in value_candidates
            if column in df.columns
        ),
        None,
    )

    if "date" not in df.columns or value_column is None:
        return pd.DataFrame(columns=["date", "tvl_usd"])

    output = df[["date", value_column]].copy()
    output = output.rename(columns={value_column: "tvl_usd"})

    output["date"] = parse_dates(output["date"])
    output["tvl_usd"] = pd.to_numeric(
        output["tvl_usd"],
        errors="coerce",
    )

    return (
        output
        .dropna(subset=["date", "tvl_usd"])
        .drop_duplicates(subset=["date"])
        .sort_values("date")
        .reset_index(drop=True)
    )


def extract_tvl_history(protocol_json: dict):
    """
    Prefer Ethereum-chain TVL.
    Fall back to overall protocol TVL only if Ethereum-specific data is unavailable.
    """

    chain_tvls = protocol_json.get("chainTvls", {})
    ethereum_entry = (
        chain_tvls.get("Ethereum")
        if isinstance(chain_tvls, dict)
        else None
    )

    if isinstance(ethereum_entry, list):
        ethereum_history = normalise_tvl_history(ethereum_entry)

        if not ethereum_history.empty:
            return ethereum_history, "ethereum_chain"

    if isinstance(ethereum_entry, dict):
        for key in ["tvl", "tokensInUsd", "tokens"]:
            ethereum_history = normalise_tvl_history(
                ethereum_entry.get(key)
            )

            if not ethereum_history.empty:
                return ethereum_history, "ethereum_chain"

    overall_history = normalise_tvl_history(protocol_json.get("tvl"))

    if not overall_history.empty:
        return overall_history, "overall_protocol_fallback"

    return pd.DataFrame(columns=["date", "tvl_usd"]), "unavailable"


def fetch_protocol_tvl_history(slug: str):
    url = f"{TVL_API}/protocol/{slug}"
    protocol_json = get_json(url)

    tvl_history, tvl_scope = extract_tvl_history(protocol_json)

    return tvl_history, tvl_scope, url


def fetch_yield_pool_catalog() -> pd.DataFrame:
    """
    Download the yield-pool catalog once.
    We then filter it locally for each selected token.
    """

    payload = get_json(f"{YIELD_API}/pools")
    records = payload.get("data", payload)

    pools_df = pd.DataFrame(records)

    required_columns = {
        "pool",
        "project",
        "symbol",
        "chain",
        "tvlUsd",
    }

    missing_columns = required_columns.difference(pools_df.columns)

    assert not missing_columns, (
        "Unexpected DeFiLlama yield-pool schema. Missing: "
        f"{sorted(missing_columns)}"
    )

    optional_columns = [
        "apy",
        "apyBase",
        "apyReward",
        "exposure",
        "ilRisk",
        "isIntrinsicSource",
    ]

    for column in optional_columns:
        if column not in pools_df.columns:
            pools_df[column] = pd.NA

    for column in ["tvlUsd", "apy", "apyBase", "apyReward"]:
        pools_df[column] = pd.to_numeric(
            pools_df[column],
            errors="coerce",
        )

    # If only APY components exist, reconstruct total APY.
    pools_df["apy"] = pools_df["apy"].fillna(
        pools_df[["apyBase", "apyReward"]].sum(
            axis=1,
            min_count=1,
        )
    )

    return pools_df


def candidate_pools_for_product(
    pools_df: pd.DataFrame,
    yield_project: str,
    token_pattern: str,
) -> pd.DataFrame:
    """
    Find pools for exactly:
    - Ethereum,
    - the intended DeFiLlama project,
    - the intended receipt token.
    """

    accepted_projects = {
        normalise_label(project)
        for project in str(yield_project).split("|")
    }

    accepted_symbols = {
        normalise_label(token)
        for token in str(token_pattern).split("|")
    }

    candidates = pools_df.copy()

    candidates["_project"] = candidates["project"].map(
        normalise_label
    )
    candidates["_symbol"] = candidates["symbol"].map(
        normalise_label
    )
    candidates["_chain"] = candidates["chain"].map(
        normalise_label
    )

    candidates = candidates.loc[
        (candidates["_chain"] == "ethereum")
        & (candidates["_project"].isin(accepted_projects))
        & (candidates["_symbol"].isin(accepted_symbols))
    ].copy()

    if candidates.empty:
        return pd.DataFrame()

    candidates["_intrinsic_source"] = (
        candidates["isIntrinsicSource"]
        .astype(str)
        .str.lower()
        .isin(["true", "1", "yes"])
    )

    candidates["_single_asset"] = (
        candidates["exposure"]
        .fillna("")
        .astype(str)
        .str.lower()
        .eq("single")
    )

    candidates["_no_il_risk"] = (
        candidates["ilRisk"]
        .fillna("")
        .astype(str)
        .str.lower()
        .isin(["no", "false"])
    )

    candidates = (
        candidates
        .sort_values(
            [
                "_intrinsic_source",
                "_single_asset",
                "_no_il_risk",
                "tvlUsd",
            ],
            ascending=[False, False, False, False],
            na_position="last",
        )
        .reset_index(drop=True)
    )

    candidates["selected"] = False
    candidates.loc[0, "selected"] = True

    candidates["selection_rule"] = (
        "Exact Ethereum project + receipt-token match; "
        "then prefer intrinsic source, single asset, "
        "no impermanent-loss risk, and highest TVL."
    )

    return candidates


def normalise_yield_history(payload) -> pd.DataFrame:
    """Convert one pool's APY history to a standard dataset."""

    records = (
        payload.get("data", payload)
        if isinstance(payload, dict)
        else payload
    )

    if not isinstance(records, list) or not records:
        return pd.DataFrame(
            columns=[
                "date",
                "apy_total",
                "apy_base",
                "apy_reward",
                "pool_tvl_usd",
            ]
        )

    df = pd.DataFrame(records)

    date_column = (
        "timestamp"
        if "timestamp" in df.columns
        else "date"
        if "date" in df.columns
        else None
    )

    if date_column is None:
        return pd.DataFrame(
            columns=[
                "date",
                "apy_total",
                "apy_base",
                "apy_reward",
                "pool_tvl_usd",
            ]
        )

    output = pd.DataFrame()
    output["date"] = parse_dates(df[date_column])

    output["apy_total"] = pd.to_numeric(
        df["apy"],
        errors="coerce",
    ) if "apy" in df.columns else pd.NA

    output["apy_base"] = pd.to_numeric(
        df["apyBase"],
        errors="coerce",
    ) if "apyBase" in df.columns else pd.NA

    output["apy_reward"] = pd.to_numeric(
        df["apyReward"],
        errors="coerce",
    ) if "apyReward" in df.columns else pd.NA

    output["pool_tvl_usd"] = pd.to_numeric(
        df["tvlUsd"],
        errors="coerce",
    ) if "tvlUsd" in df.columns else pd.NA

    output["apy_total"] = output["apy_total"].fillna(
        output[["apy_base", "apy_reward"]].sum(
            axis=1,
            min_count=1,
        )
    )

    return (
        output
        .dropna(subset=["date", "apy_total"])
        .drop_duplicates(subset=["date"])
        .sort_values("date")
        .reset_index(drop=True)
    )


def fetch_yield_history(pool_id: str):
    url = f"{YIELD_API}/chart/{pool_id}"
    payload = get_json(url)

    return normalise_yield_history(payload), url

In [4]:
yield_catalog_df = fetch_yield_pool_catalog()

tvl_frames = []
yield_frames = []
candidate_frames = []
summary_rows = []

for _, product in protocols_df.iterrows():

    product_id = product["product_id"]
    protocol = product["protocol"]
    category = product["category"]
    token = product["token"]
    slug = product["defillama_slug"]
    yield_project = product["yield_project"]
    yield_symbol = product["yield_symbol"]

    # ---------------------------------------------------------
    # TVL history
    # ---------------------------------------------------------

    tvl_status = "ok"
    tvl_error = pd.NA
    tvl_scope = pd.NA
    tvl_source_url = f"{TVL_API}/protocol/{slug}"

    latest_tvl_usd = pd.NA
    latest_tvl_date = pd.NaT

    try:
        tvl_history, tvl_scope, tvl_source_url = (
            fetch_protocol_tvl_history(slug)
        )

        if tvl_history.empty:
            tvl_status = "empty_history"

        else:
            tvl_history["product_id"] = product_id
            tvl_history["protocol"] = protocol
            tvl_history["category"] = category
            tvl_history["token"] = token
            tvl_history["defillama_slug"] = slug
            tvl_history["tvl_scope"] = tvl_scope
            tvl_history["source_url"] = tvl_source_url
            tvl_history["retrieved_at"] = RETRIEVED_AT

            tvl_frames.append(tvl_history)

            latest_tvl = tvl_history.iloc[-1]
            latest_tvl_usd = latest_tvl["tvl_usd"]
            latest_tvl_date = latest_tvl["date"]

    except (requests.RequestException, ValueError, KeyError) as error:
        tvl_status = "failed"
        tvl_scope = "unavailable"
        tvl_error = str(error)

    # ---------------------------------------------------------
    # Yield history
    # ---------------------------------------------------------

    yield_status = "no_matching_pool"
    yield_error = pd.NA

    selected_pool_id = pd.NA
    selected_pool_symbol = pd.NA
    selected_pool_source = pd.NA

    latest_apy_total = pd.NA
    latest_apy_base = pd.NA
    latest_apy_reward = pd.NA
    latest_yield_date = pd.NaT

    candidates = candidate_pools_for_product(
        pools_df=yield_catalog_df,
        yield_project=yield_project,
        token_pattern=yield_symbol,
    )

    if not candidates.empty:

        candidate_export = candidates[
            [
                "pool",
                "project",
                "symbol",
                "chain",
                "tvlUsd",
                "apy",
                "apyBase",
                "apyReward",
                "exposure",
                "ilRisk",
                "isIntrinsicSource",
                "selected",
                "selection_rule",
            ]
        ].copy()

        candidate_export["product_id"] = product_id
        candidate_export["protocol"] = protocol
        candidate_export["category"] = category
        candidate_export["token"] = token
        candidate_export["yield_project"] = yield_project
        candidate_export["retrieved_at"] = RETRIEVED_AT

        candidate_frames.append(candidate_export)

        selected_pool = candidates.loc[
            candidates["selected"]
        ].iloc[0]

        selected_pool_id = selected_pool["pool"]
        selected_pool_symbol = selected_pool["symbol"]

        try:
            yield_history, selected_pool_source = fetch_yield_history(
                selected_pool_id
            )

            if yield_history.empty:
                yield_status = "empty_history"

            else:
                yield_status = "ok"

                yield_history["product_id"] = product_id
                yield_history["protocol"] = protocol
                yield_history["category"] = category
                yield_history["token"] = token
                yield_history["pool_id"] = selected_pool_id
                yield_history["pool_symbol"] = selected_pool_symbol
                yield_history["source_url"] = selected_pool_source
                yield_history["retrieved_at"] = RETRIEVED_AT

                yield_frames.append(yield_history)

                latest_yield = yield_history.iloc[-1]
                latest_apy_total = latest_yield["apy_total"]
                latest_apy_base = latest_yield["apy_base"]
                latest_apy_reward = latest_yield["apy_reward"]
                latest_yield_date = latest_yield["date"]

        except (requests.RequestException, ValueError, KeyError) as error:
            yield_status = "failed"
            yield_error = str(error)

    summary_rows.append(
        {
            "product_id": product_id,
            "protocol": protocol,
            "category": category,
            "token": token,
            "defillama_slug": slug,
            "yield_project": yield_project,
            "yield_symbol": yield_symbol,
            "tvl_status": tvl_status,
            "tvl_scope": tvl_scope,
            "tvl_error": tvl_error,
            "latest_tvl_usd": latest_tvl_usd,
            "latest_tvl_date": latest_tvl_date,
            "yield_status": yield_status,
            "yield_error": yield_error,
            "selected_pool_id": selected_pool_id,
            "selected_pool_symbol": selected_pool_symbol,
            "latest_apy_total": latest_apy_total,
            "latest_apy_base": latest_apy_base,
            "latest_apy_reward": latest_apy_reward,
            "latest_yield_date": latest_yield_date,
            "retrieved_at": RETRIEVED_AT,
        }
    )

In [5]:
tvl_columns = [
    "date",
    "tvl_usd",
    "product_id",
    "protocol",
    "category",
    "token",
    "defillama_slug",
    "tvl_scope",
    "source_url",
    "retrieved_at",
]

yield_columns = [
    "date",
    "apy_total",
    "apy_base",
    "apy_reward",
    "pool_tvl_usd",
    "product_id",
    "protocol",
    "category",
    "token",
    "pool_id",
    "pool_symbol",
    "source_url",
    "retrieved_at",
]

candidate_columns = [
    "pool",
    "project",
    "symbol",
    "chain",
    "tvlUsd",
    "apy",
    "apyBase",
    "apyReward",
    "exposure",
    "ilRisk",
    "isIntrinsicSource",
    "selected",
    "selection_rule",
    "product_id",
    "protocol",
    "category",
    "token",
    "yield_project",
    "retrieved_at",
]

staking_tvl_timeseries_df = (
    pd.concat(tvl_frames, ignore_index=True)
    if tvl_frames
    else pd.DataFrame(columns=tvl_columns)
)

staking_yield_timeseries_df = (
    pd.concat(yield_frames, ignore_index=True)
    if yield_frames
    else pd.DataFrame(columns=yield_columns)
)

yield_pool_candidates_df = (
    pd.concat(candidate_frames, ignore_index=True)
    if candidate_frames
    else pd.DataFrame(columns=candidate_columns)
)

staking_protocol_summary_df = pd.DataFrame(summary_rows)

if not staking_tvl_timeseries_df.empty:
    staking_tvl_timeseries_df = (
        staking_tvl_timeseries_df
        .sort_values(["product_id", "date"])
        .reset_index(drop=True)
    )

if not staking_yield_timeseries_df.empty:
    staking_yield_timeseries_df = (
        staking_yield_timeseries_df
        .sort_values(["product_id", "date"])
        .reset_index(drop=True)
    )

output_paths = {
    "TVL history": DATA_PROCESSED / "staking_tvl_timeseries.csv",
    "Yield history": DATA_PROCESSED / "staking_yield_timeseries.csv",
    "Yield-pool candidates": DATA_PROCESSED / "yield_pool_candidates.csv",
    "Protocol summary": DATA_PROCESSED / "staking_protocol_summary.csv",
}

staking_tvl_timeseries_df.to_csv(
    output_paths["TVL history"],
    index=False,
)

staking_yield_timeseries_df.to_csv(
    output_paths["Yield history"],
    index=False,
)

yield_pool_candidates_df.to_csv(
    output_paths["Yield-pool candidates"],
    index=False,
)

staking_protocol_summary_df.to_csv(
    output_paths["Protocol summary"],
    index=False,
)

for dataset_name, output_path in output_paths.items():
    print(f"{dataset_name}: {output_path}")

TVL history: ..\data\processed\staking_tvl_timeseries.csv
Yield history: ..\data\processed\staking_yield_timeseries.csv
Yield-pool candidates: ..\data\processed\yield_pool_candidates.csv
Protocol summary: ..\data\processed\staking_protocol_summary.csv


In [7]:
summary_display_columns = [
    "protocol",
    "category",
    "token",
    "tvl_status",
    "tvl_scope",
    "latest_tvl_usd",
    "yield_status",
    "selected_pool_symbol",
    "latest_apy_total",
]

staking_protocol_summary_df[
    summary_display_columns
]

,protocol,category,token,tvl_status,tvl_scope,latest_tvl_usd,yield_status,selected_pool_symbol,latest_apy_total
0,Lido,liquid_staking,stETH,ok,ethereum_chain,14210649573,ok,STETH,2.37900
1,Rocket Pool,liquid_staking,rETH,ok,ethereum_chain,835959774,ok,RETH,1.95140
2,Ether.fi,liquid_restaking,weETH,ok,ethereum_chain,2618471260,ok,WEETH,0.07906
3,Renzo,liquid_restaking,ezETH,ok,ethereum_chain,61167309,ok,EZETH,1.74728
4,Kelp DAO,liquid_restaking,rsETH,ok,ethereum_chain,876723859,ok,RSETH,2.41000


In [8]:
selected_pool_columns = [
    "protocol",
    "token",
    "project",
    "symbol",
    "chain",
    "pool",
    "tvlUsd",
    "apy",
    "apyBase",
    "apyReward",
    "isIntrinsicSource",
]

selected_pools_df = yield_pool_candidates_df.loc[
    yield_pool_candidates_df["selected"],
    selected_pool_columns,
]

selected_pools_df

,protocol,token,project,symbol,chain,pool,tvlUsd,apy,apyBase,apyReward,isIntrinsicSource
0,Lido,stETH,lido,STETH,Ethereum,747c1d2a-c668-4682-b9f9-296708a3dd90,14176662137,2.37900,2.37900,NaN,<NA>
1,Rocket Pool,rETH,rocket-pool,RETH,Ethereum,d4b3c522-6127-4b89-bedf-83641cdcd2eb,2121447703,1.95140,1.95140,NaN,<NA>
2,Ether.fi,weETH,ether.fi-stake,WEETH,Ethereum,46bd2bdf-6d92-4066-b482-e885ee172264,2755854236,0.07906,0.00000,0.07906,<NA>
3,Renzo,ezETH,renzo,EZETH,Ethereum,e28e32b5-e356-41d9-8dc7-a376ece56619,82097411,1.74728,1.74728,NaN,<NA>
4,Kelp DAO,rsETH,kelp,RSETH,Ethereum,33c732f6-a78d-41da-af5b-ccd9fa5e52d5,877500917,2.41000,2.41000,NaN,<NA>


In [9]:
unresolved_products_df = staking_protocol_summary_df.loc[
    staking_protocol_summary_df["yield_status"] != "ok",
    [
        "protocol",
        "token",
        "yield_project",
        "yield_symbol",
        "yield_status",
        "yield_error",
    ],
]

if unresolved_products_df.empty:
    print("All five products have a selected yield pool and APY history.")
else:
    print(
        "These products need their mapping checked. "
        "Do not replace missing APY with zero."
    )
    display(unresolved_products_df)

All five products have a selected yield pool and APY history.
